# March Madness 2026 — Bracket Prediction Engine

Four models, one bracket. Run each model through the full tournament, compare predictions week-over-week as real results come in, and crown a champion.

| Layer | Purpose |
|-------|---------|
| **1 – Source Data** | Team DB + bracket structure |
| **2 – Models** | Advanced Metrics · Animal Kingdom · Seeding Only · Vegas Odds |
| **3 – Evaluation** | Round-by-round accuracy matrix |
| **4 – Fun Zone** | Full bracket with W/L predictions |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

DATA_DIR = ROOT / "data"

---
## Layer 1 — Source Data & Bracket Structure

Load the team database (Barttorvik metrics, coach stats, W/L records) and build the 2026 bracket from Kaggle seeds + slot template.

In [ ]:
from engine.db import TeamDB
from engine.bracket import Bracket, ROUND_LABELS

db = TeamDB(DATA_DIR)

seeds_df = pd.read_csv(DATA_DIR / "kaggle" / "MNCAATourneySeeds.csv")
slots_df = pd.read_csv(DATA_DIR / "kaggle" / "MNCAATourneySlots.csv")
db.load_seeds(seeds_df, season=2026)

print(f"TeamDB loaded: {len(db._team_facts)} teams, {len(db._seeds)} seeds")
print(f"Bracket slot template: {len(slots_df[slots_df['Season'] == slots_df['Season'].max()])} slots")

---
## Layer 2 — Models

Initialize all four prediction models. Each implements the same interface: `predict(team_a, team_b, db) → {team_a_score, team_b_score, winner_id, confidence}`.

> **Prerequisites**
> - **Advanced Metrics**: run `march_madness.ipynb` through §2.7 to export trained regressors to `data/models/`.
> - **Animal Kingdom**: set `ANTHROPIC_API_KEY` env var (results are cached after first run).
> - **Vegas Odds**: optionally place real lines in `data/vegas_lines.csv`. Falls back to AI-estimated lines when real lines are missing (needs `ANTHROPIC_API_KEY`; results cached). Estimated lines are flagged in output.
> - **Seeding Only**: no prerequisites.

In [ ]:
from engine.models.seeding import SeedingModel
from engine.models.advanced_metrics import AdvancedMetricsModel
from engine.models.animal_kingdom import AnimalKingdomModel
from engine.models.vegas_odds import VegasOddsModel

models = {}

# --- Seeding (always available) ---
models["Seeding Only"] = SeedingModel(db=db)
print("[OK] Seeding Only model loaded")

# --- Advanced Metrics (needs exported pkl files) ---
models_dir = DATA_DIR / "models"
if (models_dir / "score_margin_model.pkl").exists():
    models["Advanced Metrics"] = AdvancedMetricsModel(models_dir)
    print("[OK] Advanced Metrics model loaded")
else:
    print("[SKIP] Advanced Metrics — run march_madness.ipynb §2.7 first to export models")

# --- Animal Kingdom (needs API key) ---
import os
if os.environ.get("ANTHROPIC_API_KEY"):
    models["Animal Kingdom"] = AnimalKingdomModel()
    print("[OK] Animal Kingdom model loaded")
else:
    print("[SKIP] Animal Kingdom — set ANTHROPIC_API_KEY env var to enable")

# --- Vegas Odds (CSV lines + optional AI fallback) ---
vegas_lines_path = DATA_DIR / "vegas_lines.csv"
vegas = VegasOddsModel(lines_path=vegas_lines_path)
has_csv = vegas_lines_path.exists()
has_api = bool(os.environ.get("ANTHROPIC_API_KEY"))
if has_csv or has_api:
    models["Vegas Odds"] = vegas
    parts = []
    if has_csv:
        parts.append(f"CSV lines from {vegas_lines_path.name}")
    if has_api:
        parts.append("AI fallback for missing lines")
    print(f"[OK] Vegas Odds model loaded ({', '.join(parts)})")
else:
    print("[SKIP] Vegas Odds — provide data/vegas_lines.csv or set ANTHROPIC_API_KEY for AI estimates")

print(f"\n{len(models)} model(s) active: {list(models.keys())}")

### Simulate: Run each model through the full bracket

In [ ]:
brackets: dict[str, Bracket] = {}
bracket_dfs: dict[str, pd.DataFrame] = {}

for name, model in models.items():
    bracket = Bracket(seeds_df, slots_df, season=2026)
    bracket.simulate(model, db)
    brackets[name] = bracket
    bracket_dfs[name] = bracket.to_dataframe(db)
    champ = bracket.get_champion(db)
    print(f"  {name:20s} → Champion: {champ}")

In [ ]:
if "Vegas Odds" in models:
    v = models["Vegas Odds"]
    total = v.real_count + v.estimated_count
    if v.estimated_count > 0:
        print(f"⚠  Vegas Odds used AI-ESTIMATED lines for {v.estimated_count} of {total} matchups")
        print(f"   (Real lines: {v.real_count} | Estimated: {v.estimated_count})")
        print(f"   To reduce estimates, add real lines to data/vegas_lines.csv")
    else:
        print(f"Vegas Odds: all {total} matchups used real lines from CSV")

### Bracket Predictions — Round of 64 matchups

In [ ]:
def show_round(bracket_dfs, round_num, cols=None):
    """Display a side-by-side summary for a given round across all models."""
    if cols is None:
        cols = ["slot_id", "region", "strong_team", "strong_seed", "weak_team", "weak_seed",
                "strong_pred_score", "weak_pred_score", "winner"]
    for name, df in bracket_dfs.items():
        rnd = df[df["round_num"] == round_num][cols].copy()
        display_cols = [c for c in cols if c in rnd.columns]
        print(f"\n{'='*60}")
        print(f"  {name}  —  {ROUND_LABELS.get(round_num, f'R{round_num}')}")
        print(f"{'='*60}")
        with pd.option_context("display.max_rows", 40, "display.width", 140):
            print(rnd[display_cols].to_string(index=False))

show_round(bracket_dfs, round_num=1)

### Full path to championship — one model at a time

In [ ]:
for name, df in bracket_dfs.items():
    print(f"\n{'#'*60}")
    print(f"  {name}")
    print(f"{'#'*60}")
    for rnd in sorted(df["round_num"].unique()):
        label = ROUND_LABELS.get(rnd, f"R{rnd}")
        rnd_df = df[df["round_num"] == rnd]
        print(f"\n--- {label} ({len(rnd_df)} games) ---")
        for _, row in rnd_df.iterrows():
            s_seed = int(row['strong_seed']) if pd.notna(row['strong_seed']) else '?'
            w_seed = int(row['weak_seed']) if pd.notna(row['weak_seed']) else '?'
            s_score = f"{row['strong_pred_score']:.0f}" if pd.notna(row['strong_pred_score']) else "?"
            w_score = f"{row['weak_pred_score']:.0f}" if pd.notna(row['weak_pred_score']) else "?"
            marker = "◄" if row['winner'] == row['strong_team'] else "►"
            print(f"  ({s_seed:>2}) {row['strong_team']:>20s} {s_score:>3}  vs  {w_score:<3} {row['weak_team']:<20s} ({w_seed:<2})  {marker} {row['winner']}")

---
## Layer 3 — Evaluation

Once real results are available, inject them and compare each model's picks.

### How to inject actuals

Create `data/actuals.csv` using **team names** and **round names** — no slot IDs or numeric IDs needed:
```
round,winner,winner_score,loser_score
R64,Duke,82,55
R64,Connecticut,75,68
Sweet 16,Duke,74,61
...
```

See [UPDATING_ACTUALS.md](UPDATING_ACTUALS.md) for the full guide, then run the cells below.

In [ ]:
# ── Toggle: set to a real file path once results are in ──
ACTUALS_PATH = None  # e.g. DATA_DIR / "actuals.csv"

if ACTUALS_PATH and Path(ACTUALS_PATH).exists():
    from engine.actuals import load_actuals

    for name, bkt in brackets.items():
        actuals = load_actuals(ACTUALS_PATH, bkt, db)
        if not actuals.empty:
            bkt.inject_actuals(actuals)
            bkt.simulate(models[name], db)
            bracket_dfs[name] = bkt.to_dataframe(db)
            print(f"  {name}: re-simulated remaining rounds")
else:
    print("No actuals file set — skipping evaluation.  (Update ACTUALS_PATH above when results are in.)")

### Accuracy Matrix

Rows show cumulative and per-round accuracy. Columns are models.

In [ ]:
from engine.evaluation import accuracy_table, spread_accuracy_table, plot_accuracy_heatmap

acc = accuracy_table(bracket_dfs)
if acc.empty:
    print("No actual results yet — accuracy table will populate once actuals are injected.")
else:
    display(acc.style.format({c: "{:.1%}" for c in acc.columns if c != "window"}))
    plot_accuracy_heatmap(acc)

    spread = spread_accuracy_table(bracket_dfs)
    if not spread.empty:
        print("\nSpread MAE (predicted margin vs actual margin):")
        display(spread.style.format({c: "{:.1f}" for c in spread.columns if c != "window"}))

---
## Layer 4 — Fun Zone: Bracket with Predicted W/L

State-by-state bracket view. Before actuals, State 0 shows pure predictions. After each week's actuals, the bracket updates: locked results in bold, fresh predictions for remaining games.

Change `display_through_round` below to control which round is the last "actual" round shown.

In [ ]:
def print_bracket(df: pd.DataFrame, model_name: str):
    """Compact bracket printout with actual/predicted markers."""
    print(f"\n{'='*72}")
    print(f"  {model_name}")
    print(f"{'='*72}")
    for rnd in sorted(df["round_num"].unique()):
        rnd_df = df[df["round_num"] == rnd]
        label = ROUND_LABELS.get(rnd, f"R{rnd}")
        print(f"\n  {label}")
        print(f"  {'─'*68}")
        for _, row in rnd_df.iterrows():
            tag = "✓" if row["is_actual"] else "…"
            s_seed = int(row['strong_seed']) if pd.notna(row['strong_seed']) else '?'
            w_seed = int(row['weak_seed']) if pd.notna(row['weak_seed']) else '?'
            s_sc = f"{row['strong_pred_score']:.0f}" if pd.notna(row['strong_pred_score']) else "?"
            w_sc = f"{row['weak_pred_score']:.0f}" if pd.notna(row['weak_pred_score']) else "?"
            if row["is_actual"] and pd.notna(row.get("actual_strong_score")):
                s_sc = f"{row['actual_strong_score']:.0f}"
                w_sc = f"{row['actual_weak_score']:.0f}"
            win = row['winner'] if row['winner'] else "TBD"
            print(f"  [{tag}] ({s_seed:>2}) {row['strong_team']:>20s} {s_sc:>3}  –  {w_sc:<3} {row['weak_team']:<20s} ({w_seed:<2})  → {win}")

for name, df in bracket_dfs.items():
    print_bracket(df, name)

---
## Weekly Iteration Cheat Sheet

Each week when new results come in:

1. **Update actuals CSV** — add rows for newly completed games.
2. **Set `ACTUALS_PATH`** in the Layer 3 cell above.
3. **Re-run from Layer 3 onward** — actuals are injected, remaining rounds re-predicted, evaluation matrix updated.

The bracket stores `pred_winner_id` (original prediction) separately from the resolved winner, so evaluation accuracy is always measured against the model's *original* pick for that slot.

### Model comparison — side-by-side picks for each round

In [ ]:
def comparison_table(bracket_dfs, round_num):
    """Build a single table showing each model's winner pick for a round."""
    ref_name = list(bracket_dfs.keys())[0]
    ref = bracket_dfs[ref_name]
    rnd = ref[ref["round_num"] == round_num][["slot_id", "region", "strong_team", "strong_seed", "weak_team", "weak_seed"]].copy()

    for name, df in bracket_dfs.items():
        rdf = df[df["round_num"] == round_num][["slot_id", "winner"]].rename(columns={"winner": name})
        rnd = rnd.merge(rdf, on="slot_id", how="left")
    return rnd

for rnd_num in range(1, 7):
    label = ROUND_LABELS.get(rnd_num, f"R{rnd_num}")
    ct = comparison_table(bracket_dfs, rnd_num)
    if ct.empty:
        continue
    print(f"\n{'='*60}")
    print(f"  {label} — Model Picks")
    print(f"{'='*60}")
    with pd.option_context("display.max_rows", 40, "display.width", 160, "display.max_columns", 15):
        print(ct.to_string(index=False))